# wf_v5.1 GPU session (diary Nov 24, task R5): the full rerun with the adopted switches
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v5.1`, folder `step1a/`. **wf-v5.1 must be pushed before this run** (cell 1 clones it;
only after the user's OK, Claude does not push). Results in `results/wf_v51/r5/` (new folders only).

Model: Llama-3.1-8B-Instruct, bf16, vLLM, temperature 0, guided JSON (setting `llama8b`), batch-invariant mode on, engine seed 0 (as wf_v5.0).
Runtime: **A100 (40 GB or more)**; cell 0 checks it. One GPU session, model load about 5 min, then about 10 min of model time (estimate).

Per set (dev50, dev50b), everything guided; a guided call with `finish_reason` "error" or a row that is not guided stops the cell (no unguided fallback):
- **Cell 7 smoke test:** one tiny guided call. If it fails, stop, copy the output and tell Claude (do not run the next cells).
- **Cell 10 Step 3** (about 550 + 475 calls): the frozen v4.4 Step 3 with **R4** (3 context snippets per cluster in the naming prompt) and **R3**
  (closed question for same-named clusters) on, from the loop-1 entities and frozen A prompts of v4.6 (in git). Stops at MAX_CALLS 900 per set.
- **Cell 11 rename:** context split big40 (v5.0) on the new columns, a G4-style rename of the changed columns (with R4 snippets), then the R3 clash
  question on columns that share a name (v5.0 named both train time parts "departure time").
- **Cell 12 loop 2 (CPU):** **replayed, not re-asked**: the wf_v4.8 W2 replay of the cached v4.7 z4 run-1 loop-2 files on the new loop-1 columns,
  with the laptop bge-m3 vectors (`results/wf_v51/r5/<set>/vectors.npz`; a text without a laptop vector goes to the GPU bge-m3 and is counted).
- **Cell 13 Step 4:** v5.0 frozen = variant **ARMg** (A + re-ask r1 + r2 + same-entity merge), seeds 0-2 -> `results/wf_v51/r5/<set>/step4_ARMg/`.
- **Cell 14:** `results/wf_v51/r5/settings.json` (vLLM settings, session time, calls, tokens, finish reasons, guided share) and
  `calls.csv`; then **Runtime > Disconnect and delete runtime**, and tell Claude "GPU done".
- Cells 1-6: setup, as in wf_v5.0. Cell 2 installs, then restart the session, rerun cells 0-1 and skip cell 2.
  Cell 3 (models) must run before any other cell imports `s1a`.
- After a crash: rerun cells 0-1, 3-9 and the cell that failed (answered calls are cached per folder).
- Colab Secrets `HF_TOKEN` and `GH_TOKEN`, notebook access on. Never put a token in a cell.


In [ ]:
#@title 0. GPU check: nvidia-smi, assert an A100 with 40 GB or more
import re, subprocess
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(smi.stdout.strip() or smi.stderr.strip())
assert smi.returncode == 0 and smi.stdout.strip(), "no GPU: set the runtime type to A100"
name, mib = [x.strip() for x in smi.stdout.splitlines()[0].split(",")]
assert re.search(r"A100|H100|H200|B200", name) and int(mib) >= 40000, f"need an A100 with 40 GB or more, got {name} {mib} MiB"
print("GPU ok:", name, mib, "MiB")


In [ ]:
#@title 1. Tokens, clone/pull wf-v5.1, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v5.1"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Data and the loop-2 models (MultiWOZ zip + ids from HF, GLiNER, deberta-v3-base, bge-m3; pinned revisions)
import json, os
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download
MODELS = Path("/content/models")
os.environ["CHK_MODELS"] = str(MODELS)          # read by chunking/chk/config.py and s1a/spans.py at import
DATA = Path("/content/attribute-extraction/chunking/data")
for f in ("MULTIWOZ2.4.zip", "conversation_ids.txt"):
    if not (DATA / f).exists():
        hf_hub_download("database-lab/attribute-extraction-data", f"chunking/{f}", repo_type="dataset", local_dir=DATA.parent)
snapshot_download("urchade/gliner_medium-v2.1", revision="40ec419335d09393f298636f471328b722c6da9e", local_dir=MODELS / "gliner_medium-v2.1")
snapshot_download("microsoft/deberta-v3-base", revision="8ccc9b6f36199bec6961081d44eb72fb3f7353f3", local_dir=MODELS / "deberta-v3-base",
                  allow_patterns=["config.json", "spm.model", "tokenizer_config.json"])
snapshot_download("BAAI/bge-m3", revision="5617a9f61b028005a4858fdac845db406aefb181", local_dir=MODELS / "bge-m3")
cfg_path = MODELS / "gliner_medium-v2.1" / "gliner_config.json"   # point GLiNER at the local backbone, as on the laptop
cfg = json.loads(cfg_path.read_text())
cfg["model_name"] = str(MODELS / "deberta-v3-base")
cfg_path.write_text(json.dumps(cfg, indent=2))
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a"); sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
from s1a import spans as SP
g = SP.load()
print("GLiNER ok:", g.inference(["I need a hotel with free wifi."], ["internet access"], threshold=0.3))
del g

In [ ]:
#@title 4. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "llama8b"  #@param ["llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 5. Load the model (gpu_memory_utilization leaves room for bge-m3)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.6  #@param {type:"number"}
BATCH_INVARIANT = True  #@param {type:"boolean"}
import os
os.environ["VLLM_BATCH_INVARIANT"] = "1" if BATCH_INVARIANT else "0"   # read when vllm is imported
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
import time
_t0 = _T_SESSION = time.time()   # the GPU session clock starts with the model load
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)
from s1a import log_v44 as L44
print("model load:", L44.save_load_time("results/wf_v51/r5/model_load.json", time.time() - _t0, MODEL, REVISION))


In [ ]:
#@title 6. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/log_v44.VllmBackend44)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
from s1a import log_v44 as L44
backend = L44.VllmBackend44(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 7. Guided smoke test (must print finish stop and {"a": "x"} or {"a": "y"}); stops the session if guided decoding fails
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import gpu_v51 as G
fin, txt = G.smoke_test(llm, sampling, backend.chat)
print("finish:", fin, "| text:", txt)


In [ ]:
#@title 8. Data check: inputs from git (loop-1 entities and A prompts, loop-2 files, Step 1a-2 folders, laptop vectors, configs)
missing = G.need()
print("missing:", missing or "none")
assert not missing, "files missing: stop and tell Claude"
print({"switches": G.SWITCHES, "context split": G.CTX, "step 4": G.G48.config(G.VARIANT), "limits": G.LIMITS})


In [ ]:
#@title 9. Encoders: bge-m3 for the Step 3 names (merge booster) and for texts without a laptop vector; laptop vectors first
from s1a import step3_v44 as S3
from run import encoder
enc3 = S3.bge_m3_encoder()
enc_1a = encoder()
vec_enc = G.G50.CachedEnc(G.R5, "vectors.npz", fallback=enc_1a)
print("encoders ready")


In [ ]:
#@title 10. Step 3 with R3 + R4 on (names, split, R3 clash, reconcile, dupcheck) -> results/wf_v51/r5/<set>/step3
MAX_CALLS = 900  #@param {type:"integer"}
import subprocess
res3 = G.step3_stage(backend, enc3, max_calls=MAX_CALLS)
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v51 R5 Step 3", "wf_v51"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 11. Context split big40, G4-style rename (R4 snippets), column clash check -> results/wf_v51/r5/rename/<set>
MAX_CALLS = 60  #@param {type:"integer"}
import subprocess
res_rename = G.rename_stage(backend, vec_enc, max_calls=MAX_CALLS)
for dev, r in res_rename.items():
    print(dev, "renamed:", [(x["entity"], x["cid"], x["old"], "->", x["new"]) for x in r["renamed"] if x["renamed"]])
    print(dev, "clash renames:", [(x["entity"], x["cid"], x["old"], "->", x["new"]) for x in r["clash"]])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v51 R5 rename and clash", "wf_v51"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 12. Loop 2 (frozen; replayed on CPU, not re-asked) and the Step 4 input -> results/wf_v51/r5/<set>/input
import json
paths = G.loop2_stage(vec_enc)
for dev, p in paths.items():
    print(dev, p, len(json.loads(p.read_text(encoding="utf-8"))), "columns")
print("texts without a laptop vector (GPU bge-m3):", vec_enc.fallback_texts)


In [ ]:
#@title 13. Step 4 v5.0 frozen (variant ARMg), seeds 0-2, dev50 and dev50b -> results/wf_v51/r5/<set>/step4_ARMg
MAX_CALLS = 140  #@param {type:"integer"}
import subprocess
res4 = G.step4_stage(backend, max_calls=MAX_CALLS)
for (var, dev), rows in res4.items():
    print(var, dev, [(x["seed"], x["ok"], x["tables"], x["error"]) for x in rows])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v51 R5 Step 4 variant ARMg", "wf_v51"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 14. vLLM settings, call counts, GPU session time -> results/wf_v51/r5/settings.json, then disconnect
import json, os, time, subprocess
def _seed():
    try:
        return llm.llm_engine.model_config.seed
    except Exception as e:
        return f"not readable: {type(e).__name__}"
cost = G.cost_report()
info = {"vllm": vllm.__version__, "batch": BATCH, "batch_invariant_env": os.environ.get("VLLM_BATCH_INVARIANT"), "engine_seed": _seed(),
        "temperature": 0, "model": MODEL, "revision": REVISION, "smoke": [fin, txt], "switches": G.SWITCHES, "context_split": G.CTX,
        "step4_variant": G.VARIANT, "loop2": "replayed on CPU (W2 replay of the cached v4.7 z4 run-1 loop-2 files), not re-asked",
        "chains": {f"{v} {d}": [(x["run"], x["ok"], x["tables"]) for x in rows] for (v, d), rows in res4.items()},
        "calls": cost, "total_calls": {s: sum(x["calls"] for x in d.values()) for s, d in cost.items()},
        "total_tokens_in": sum(x["prompt_tokens"] for d in cost.values() for x in d.values()),
        "total_tokens_out": sum(x["output_tokens"] for d in cost.values() for x in d.values()),
        "guided_share": {s: {dev: (x["guided"] / x["calls"] if x["calls"] else None) for dev, x in d.items()} for s, d in cost.items()},
        "unguided_rows": {f"{s} {d}": len(G.G50.unguided_rows(f(G.R5, d))) for s, f in G.STAGES for d in G.SETS},
        "laptop_vector_fallback_texts": vec_enc.fallback_texts,
        "gpu_session_seconds": round(time.time() - _T_SESSION, 1), "model_load_seconds": L44.load_time("results/wf_v51/r5/model_load.json")}
open("results/wf_v51/r5/settings.json", "w").write(json.dumps(info, indent=1, default=str) + "\n")
print(json.dumps(info, indent=1, default=str))
assert not any(info["unguided_rows"].values()), "some rows are not guided: tell Claude"
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v51 R5 settings and session time", "wf_v51"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("Done: Runtime > Disconnect and delete runtime, then tell Claude 'GPU done'")
